# 🏥 ClaimMitra Security Series
## Day 01 — Identity & Action Boundary with OPA
---
**Program:** AI Agent Security & Governance | **Module 4**
**Format:** 30 min demo → 45 min hands-on → 15 min discussion
**Tool today:** [Open Policy Agent (OPA)](https://www.openpolicyagent.org/) — open source, no cost

> **How to use this notebook**
> Cells marked 🎬 are the instructor demo. Cells marked 🛠️ are your hands-on exercises.
> All base-agent cells (marked 🔨) are identical across every day of this series.

## 📖 The ClaimMitra Story

**ClaimMitra** is an AI-powered insurance claims assistant deployed by a mid-size InsurTech firm.
It handles ~12,000 interactions a day across a web portal and an internal claims-desk interface.

The agent can:
| Tool | What it does |
|------|-------------|
| `get_policy_details` | Fetch policy coverage and limits |
| `file_claim` | Open a new claim against a policy |
| `check_claim_status` | Return current status + timeline |
| `approve_settlement` | Disburse a settlement amount |
| `get_customer_pii` | Retrieve raw customer identity data |

**Users of the system:**

| Role | Who | Approval limit |
|------|-----|---------------|
| `customer` | Policyholders (web portal) | None |
| `agent` | Junior claims desk staff | ₹10,000 |
| `senior_agent` | Experienced claims handlers | ₹50,000 |
| `manager` | Claims managers | ₹2,00,000 |
| `compliance_officer` | Audit & fraud team | PII access |

When the program started (Day 0), ClaimMitra had only **one security layer: a login gate.**
JWT token present → access granted. No check on *what* the logged-in user was allowed to do.

## ⚠️ The Incident — October 15th

> *Names are fictional. The attack pattern is real.*

**Rajan Mehta**, a junior claims agent (role: `agent`), left his laptop unattended at a café in
Koramangala, Bengaluru. A bad actor accessed his saved browser session — still authenticated
to the ClaimMitra portal.

**What happened next, in 22 minutes:**

```
10:43  Attacker opens ClaimMitra portal with Rajan's live session
10:44  Queries claim CLM-2024-8821 (amount: ₹4,20,000 — pending manager approval)
10:46  Calls approve_settlement(claim_id="CLM-2024-8821", amount=420000)
10:46  ✅ Agent executes. Settlement disbursed.
10:51  Calls approve_settlement(claim_id="CLM-2024-8831", amount=27000)   ← second fraud
10:58  Calls approve_settlement(claim_id="CLM-2024-8809", amount=23000)   ← third fraud
11:05  Security team notices anomalous wire transfers. Session killed.
       Total disbursed fraudulently: ₹4,70,000
```

**Root cause:** The agent framework checked *authentication* (is this a valid JWT?) but not
*authorisation at the action level* (is this role allowed to call this tool with these parameters?).
Rajan's role was `agent`. Agents should never approve settlements > ₹10,000.
The agent didn't know that. There was no policy to enforce it.

---
### 🔑 Today's Fix: Move Policy Out of Code, Into OPA

**Interesting fact:** OPA was born at Styra in 2016, originally to solve exactly this problem
at scale — Kubernetes admission control. The name "Rego" (the policy language) comes from
*"regulation"*. Today it's a CNCF graduated project used by Netflix, Goldman Sachs, and
Atlassian to enforce fine-grained policies across microservices and AI agents alike.
The key insight: **policy as data, not code** — you can change a Rego rule without redeploying
your application.

## 🧠 Concept: Identity & Action Boundary

An **action boundary** is a policy that answers three questions *before* any tool executes:

1. **Who** is making the request? (identity — from JWT, session, API key)
2. **What** are they trying to do? (action — tool name + parameters)
3. **Are they allowed to do this, right now?** (policy decision)

OPA separates this logic from your application using **Rego** — a declarative query language.

```
Application Code          OPA Policy Engine
─────────────────         ─────────────────────────────
Agent calls tool    ──►   Input: { user, role, action, amount }
(before execution)        Policy: claimmitra/authz.rego
                    ◄──   Decision: { allow: true/false, reason: "..." }
Agent executes             Application enforces the decision
(only if allowed)
```

**Why not just check roles in the application?**
Because role checks scattered across tool code become impossible to audit. OPA gives you
one place where every policy lives, versioned in Git, testable independently of the app.

## 🔨 ClaimMitra Base Agent *(same cells across all days)*

In [2]:
# Cell B1 — Install packages (run once)
%pip install -q langgraph langchain-core requests
print("✅ packages ready")

Note: you may need to restart the kernel to use updated packages.
✅ packages ready


c:\Users\l76-labuser181612\AI-Lab\AI-Agent-Security\.venv\Scripts\python.exe: No module named pip


In [3]:
# Cell B2 — Imports
import sqlite3, json, os, subprocess, time, textwrap, requests
from datetime import datetime
from typing import TypedDict, Annotated, Sequence
import operator

try:
    from langchain_core.messages import HumanMessage, AIMessage, ToolMessage, BaseMessage
    from langchain_core.tools import tool
    from langgraph.graph import StateGraph, END
    LANGGRAPH = True
except ImportError:
    LANGGRAPH = False
    print("⚠️  LangGraph not installed — agent graph will use mock mode")

    class BaseMessage:
        def __init__(self, content="", **kwargs):
            self.content = content

    class HumanMessage(BaseMessage):
        pass

    class AIMessage(BaseMessage):
        pass

    class ToolMessage(BaseMessage):
        pass

print(f"LangGraph available: {LANGGRAPH}")

LangGraph available: True


In [4]:
# Cell B3 — ClaimMitra database (SQLite in-memory, seeded with sample data)
def init_db():
    conn = sqlite3.connect(":memory:", check_same_thread=False)
    conn.row_factory = sqlite3.Row
    cur = conn.cursor()
    SQL = '''
        CREATE TABLE policies (
            policy_id TEXT PRIMARY KEY,
            customer_id TEXT,
            customer_name TEXT,
            coverage_type TEXT,
            coverage_limit REAL,
            premium REAL,
            status TEXT
        );
        CREATE TABLE claims (
            claim_id TEXT PRIMARY KEY,
            policy_id TEXT,
            amount REAL,
            description TEXT,
            status TEXT,
            filed_date TEXT,
            owner_id TEXT
        );
        CREATE TABLE customers (
            customer_id TEXT PRIMARY KEY,
            name TEXT,
            dob TEXT,
            pan TEXT,
            phone TEXT,
            email TEXT,
            address TEXT
        );
        INSERT INTO customers VALUES
            ("C001","Priya Sharma","1985-03-12","ABCPS1234A","9876543210",
             "priya@example.com","12 MG Road, Bengaluru"),
            ("C002","Rajan Mehta","1990-07-22","DEFRM5678B","9988776655",
             "rajan@example.com","45 Koramangala, Bengaluru"),
            ("C003","Sunita Rao","1978-11-05","GHISR9012C","9123456789",
             "sunita@example.com","78 Jubilee Hills, Hyderabad");
        INSERT INTO policies VALUES
            ("POL-001","C001","Priya Sharma","Health",500000,12000,"active"),
            ("POL-002","C002","Rajan Mehta","Motor",200000,8500,"active"),
            ("POL-003","C003","Sunita Rao","Health",750000,18000,"active");
        INSERT INTO claims VALUES
            ("CLM-2024-8809","POL-001",23000,"Hospitalisation - Apollo, Sep 24",
             "pending","2024-09-28","C001"),
            ("CLM-2024-8821","POL-003",420000,"Cardiac surgery - KIMS, Oct 1",
             "pending","2024-10-01","C003"),
            ("CLM-2024-8831","POL-002",27000,"Rear-end collision, Oct 5",
             "pending","2024-10-05","C002");
    '''
    cur.executescript(SQL)
    conn.commit()
    return conn

DB = init_db()
print("✅ Database initialised — 3 customers, 3 policies, 3 claims")

✅ Database initialised — 3 customers, 3 policies, 3 claims


In [5]:
# Cell B4 — ClaimMitra tool definitions
def get_policy_details(policy_id: str) -> dict:
    row = DB.execute("SELECT * FROM policies WHERE policy_id=?", (policy_id,)).fetchone()
    return dict(row) if row else {"error": f"Policy {policy_id} not found"}

def file_claim(policy_id: str, amount: float, description: str) -> dict:
    claim_id = f"CLM-{datetime.now().strftime('%Y-%m%d%H%M%S')}"
    DB.execute(
        "INSERT INTO claims VALUES (?,?,?,?,?,?,?)",
        (claim_id, policy_id, amount, description, "filed",
         datetime.now().strftime("%Y-%m-%d"), "system")
    )
    DB.commit()
    return {"claim_id": claim_id, "status": "filed", "amount": amount}

def check_claim_status(claim_id: str) -> dict:
    row = DB.execute("SELECT * FROM claims WHERE claim_id=?", (claim_id,)).fetchone()
    return dict(row) if row else {"error": f"Claim {claim_id} not found"}

def approve_settlement(claim_id: str, amount: float) -> dict:
    DB.execute(
        "UPDATE claims SET status='approved' WHERE claim_id=?", (claim_id,)
    )
    DB.commit()
    return {"claim_id": claim_id, "status": "approved", "amount_disbursed": amount,
            "timestamp": datetime.now().isoformat()}

def get_customer_pii(customer_id: str) -> dict:
    row = DB.execute("SELECT * FROM customers WHERE customer_id=?", (customer_id,)).fetchone()
    return dict(row) if row else {"error": f"Customer {customer_id} not found"}

TOOLS = {
    "get_policy_details": get_policy_details,
    "file_claim": file_claim,
    "check_claim_status": check_claim_status,
    "approve_settlement": approve_settlement,
    "get_customer_pii": get_customer_pii,
}
print("✅ Tools registered:", list(TOOLS.keys()))

✅ Tools registered: ['get_policy_details', 'file_claim', 'check_claim_status', 'approve_settlement', 'get_customer_pii']


In [6]:
# Cell B5 — MockLLM (no API key needed; swap in real LLM if preferred)
# To use OpenAI instead:
#   from langchain_openai import ChatOpenAI
#   LLM = ChatOpenAI(model="gpt-4o-mini")
#   Then replace MockLLM() with LLM below.

class MockResponse:
    def __init__(self, content="", tool_calls=None):
        self.content = content
        self.tool_calls = tool_calls or []

class MockLLM:
    """Rule-based mock so demos work without any API key."""
    def invoke(self, messages):
        text = " ".join(
            m.content for m in messages if hasattr(m, "content") and m.content
        ).lower()
        if "approve" in text or "settle" in text:
            # extract claim id and amount from context
            return MockResponse(tool_calls=[{
                "id": "tc_approve_1",
                "name": "approve_settlement",
                "args": {"claim_id": "CLM-2024-8821", "amount": 420000}
            }])
        if "status" in text or "check" in text:
            return MockResponse(tool_calls=[{
                "id": "tc_status_1",
                "name": "check_claim_status",
                "args": {"claim_id": "CLM-2024-8821"}
            }])
        if "pii" in text or "customer details" in text or "identity" in text:
            return MockResponse(tool_calls=[{
                "id": "tc_pii_1",
                "name": "get_customer_pii",
                "args": {"customer_id": "C003"}
            }])
        return MockResponse(content="I can help with claim status, approvals, or policy details.")

LLM = MockLLM()
print("✅ MockLLM ready (no API key required)")

✅ MockLLM ready (no API key required)


In [7]:
# Cell B6 — Basic ClaimMitra agent (NO security — vulnerable baseline)
class AgentState(TypedDict):
    messages: Annotated[Sequence[BaseMessage], operator.add]
    user_id: str
    role: str

def run_agent_unprotected(user_message: str, user_id: str = "U001", role: str = "agent"):
    """Run agent with NO policy enforcement — dangerous baseline."""
    print(f"\n{'='*60}")
    print(f"User [{role}:{user_id}]: {user_message}")
    print(f"{'='*60}")

    messages = [HumanMessage(content=user_message)]
    llm_response = LLM.invoke(messages)

    if llm_response.tool_calls:
        for tc in llm_response.tool_calls:
            tool_name = tc["name"]
            tool_args = tc["args"]
            print(f"\n🔧 Agent calling: {tool_name}({tool_args})")
            result = TOOLS[tool_name](**tool_args)
            print(f"📤 Result: {json.dumps(result, indent=2)}")
    else:
        print(f"\n💬 Agent: {llm_response.content}")
    return llm_response

print("✅ Unprotected agent ready")

✅ Unprotected agent ready


## 🎬 Demo Part 1 — The Vulnerability in Action

Watch what happens when a `junior agent` (Rajan's session) requests a ₹4,20,000 settlement.
The agent has no idea this role is not allowed to approve amounts above ₹10,000.

In [8]:
# 🎬 DEMO — Attacker using Rajan's session (role=agent, limit should be ₹10,000)
run_agent_unprotected(
    user_message="Please approve the settlement for claim CLM-2024-8821 for ₹4,20,000",
    user_id="rajan_mehta",
    role="agent"   # junior agent — should NOT be allowed to approve > ₹10,000
)


User [agent:rajan_mehta]: Please approve the settlement for claim CLM-2024-8821 for ₹4,20,000

🔧 Agent calling: approve_settlement({'claim_id': 'CLM-2024-8821', 'amount': 420000})
📤 Result: {
  "claim_id": "CLM-2024-8821",
  "status": "approved",
  "amount_disbursed": 420000,
  "timestamp": "2026-10-27T05:28:37.930518"
}


## 🎬 Demo Part 2 — OPA to the Rescue

We will now:
1. Download the OPA binary (single file, ~100MB, no install needed)
2. Start OPA as a local server
3. Define our policy in Rego
4. Wrap every tool call through OPA before execution

In [9]:
# 🎬 Step 1 — Download OPA binary (Cross-platform: Windows / macOS / Linux)
import os, sys, platform, tempfile, urllib.request, subprocess

system = platform.system().lower()
machine = platform.machine().lower()
temp_dir = tempfile.gettempdir()

if system == "windows":
    opa_filename = "opa.exe"
    opa_url = "https://github.com/open-policy-agent/opa/releases/download/v0.68.0/opa_windows_amd64.exe"
elif system == "darwin":
    opa_filename = "opa"
    arch = "arm64" if "arm" in machine or "aarch64" in machine else "amd64"
    opa_url = f"https://github.com/open-policy-agent/opa/releases/download/v0.68.0/opa_darwin_{arch}"
else:
    opa_filename = "opa"
    arch = "arm64_static" if "arm" in machine or "aarch64" in machine else "amd64_static"
    opa_url = f"https://github.com/open-policy-agent/opa/releases/download/v0.68.0/opa_linux_{arch}"

OPA_PATH = os.path.join(temp_dir, opa_filename)

if not os.path.exists(OPA_PATH):
    print(f"Downloading OPA for {platform.system()} ({opa_url}) ...")
    try:
        urllib.request.urlretrieve(opa_url, OPA_PATH)
        if system != "windows":
            os.chmod(OPA_PATH, 0o755)
        print("✅ OPA downloaded")
    except Exception as e:
        print(f"⚠️  Could not download OPA: {e}")
else:
    print(f"✅ OPA binary file present at {OPA_PATH}")

# Verify execution permission (handles Windows Application Control / AppLocker restrictions)
USE_MOCK_OPA = False
try:
    result = subprocess.run([OPA_PATH, "version"], capture_output=True, text=True)
    if result.returncode == 0:
        print("✅ Live OPA binary verified:", result.stdout.split("\n")[0])
    else:
        USE_MOCK_OPA = True
        print("⚠️  OPA binary returned non-zero exit code. Enabling Python OPA fallback engine.")
except OSError as e:
    USE_MOCK_OPA = True
    print(f"⚠️  Application Control / OS security policy blocked execution of '{opa_filename}': {e}")
    print("👉 Automatically enabled Python OPA Policy Engine fallback (all demos & exercises will work without external binary!)")

✅ OPA downloaded
✅ Live OPA binary verified: Version: 0.68.0


In [10]:
# 🎬 Step 2 — Write ClaimMitra Rego policy
REGO_POLICY = textwrap.dedent('''
    package claimmitra.authz

    import future.keywords.in

    # Default deny
    default allow = false
    default reason = "Action not permitted for this role"

    # CUSTOMER: read-only on their own claims
    allow {
        input.role == "customer"
        input.action in ["check_claim_status", "get_policy_details"]
        input.user_is_owner == true
    }
    reason = "Customer can only view their own claims" {
        input.role == "customer"
        not input.user_is_owner
    }

    # AGENT: file and check, small approvals only
    allow {
        input.role == "agent"
        input.action in ["file_claim", "check_claim_status", "get_policy_details"]
    }
    allow {
        input.role == "agent"
        input.action == "approve_settlement"
        input.amount <= 10000
    }
    reason = "Agents may only approve settlements up to Rs 10,000" {
        input.role == "agent"
        input.action == "approve_settlement"
        input.amount > 10000
    }

    # SENIOR AGENT
    allow {
        input.role == "senior_agent"
        input.action in ["file_claim", "check_claim_status", "get_policy_details",
                         "approve_settlement"]
        input.amount <= 50000
    }
    reason = "Senior agents may only approve up to Rs 50,000" {
        input.role == "senior_agent"
        input.action == "approve_settlement"
        input.amount > 50000
    }

    # MANAGER
    allow {
        input.role == "manager"
        input.action in ["file_claim", "check_claim_status", "get_policy_details",
                         "approve_settlement"]
        input.amount <= 200000
    }

    # COMPLIANCE OFFICER: full access including PII
    allow {
        input.role == "compliance_officer"
    }
''')

POLICY_PATH = os.path.join(tempfile.gettempdir(), "claimmitra_authz.rego")
with open(POLICY_PATH, "w") as f:
    f.write(REGO_POLICY)
print("✅ Rego policy written to", POLICY_PATH)
print("\nPolicy preview (first 15 lines):")
print("\n".join(REGO_POLICY.strip().split("\n")[:15]))

✅ Rego policy written to C:\Users\L76-LA~1\AppData\Local\Temp\2\claimmitra_authz.rego

Policy preview (first 15 lines):
package claimmitra.authz

import future.keywords.in

# Default deny
default allow = false
default reason = "Action not permitted for this role"

# CUSTOMER: read-only on their own claims
allow {
    input.role == "customer"
    input.action in ["check_claim_status", "get_policy_details"]
    input.user_is_owner == true
}
reason = "Customer can only view their own claims" {


In [11]:
# 🎬 Step 3 — Start OPA server
OPA_PROC = None
OPA_URL_BASE = "http://localhost:8181"

def start_opa():
    global OPA_PROC, USE_MOCK_OPA
    if USE_MOCK_OPA:
        print("✅ Python OPA Policy Engine active (in-memory mode — no external binary needed)")
        return

    # Kill any existing OPA instance
    if OPA_PROC:
        OPA_PROC.terminate()
        time.sleep(1)
    try:
        OPA_PROC = subprocess.Popen(
            [OPA_PATH, "run", "--server", "--addr", "localhost:8181",
             "--log-level", "error"],
            stdout=subprocess.DEVNULL, stderr=subprocess.DEVNULL
        )
        time.sleep(2)  # wait for server to start
        r = requests.get(f"{OPA_URL_BASE}/health", timeout=3)
        if r.status_code == 200:
            print("✅ OPA server running at", OPA_URL_BASE)
        else:
            USE_MOCK_OPA = True
            print("⚠️  OPA health check failed. Switched to Python OPA engine.")
    except Exception as e:
        USE_MOCK_OPA = True
        print(f"⚠️  Could not start OPA binary ({e}). Switched to Python OPA engine.")

start_opa()

✅ OPA server running at http://localhost:8181


In [12]:
# 🎬 Step 4 — Load Rego policy into OPA
CURRENT_POLICY_TEXT = REGO_POLICY

def evaluate_python_policy(inp: dict, policy_text: str = ""):
    """In-memory policy engine fallback when external OPA binary is blocked by Windows security."""
    role = inp.get("role")
    action = inp.get("action")
    amount = inp.get("amount", 0)
    user_is_owner = inp.get("user_is_owner", False)

    # Dynamic rule check for Exercise 1: fraud_investigator
    if "fraud_investigator" in policy_text and role == "fraud_investigator":
        if action in ["get_customer_pii", "check_claim_status"]:
            return True, "Permitted"
        return False, "Fraud investigator cannot approve settlements"

    # Dynamic rule check for Exercise 2: time-based policy (business hours 9 AM to 6 PM IST)
    if ("time.clock" in policy_text or "Asia/Kolkata" in policy_text) and role == "agent" and action == "approve_settlement":
        from datetime import timezone, timedelta
        ist = timezone(timedelta(hours=5, minutes=30))
        hr = datetime.now(ist).hour
        if hr < 9 or hr >= 18:
            return False, "Agents may only approve settlements during business hours (9 AM–6 PM IST)"

    if role == "customer":
        if action in ["check_claim_status", "get_policy_details"]:
            if user_is_owner:
                return True, "Permitted"
            return False, "Customer can only view their own claims"
        return False, "Action not permitted for customer"

    if role == "agent":
        if action in ["file_claim", "check_claim_status", "get_policy_details"]:
            return True, "Permitted"
        if action == "approve_settlement":
            if amount <= 10000:
                return True, "Permitted"
            return False, "Agents may only approve settlements up to Rs 10,000"
        return False, "Action not permitted for agent"

    if role == "senior_agent":
        if action in ["file_claim", "check_claim_status", "get_policy_details"]:
            return True, "Permitted"
        if action == "approve_settlement":
            if amount <= 50000:
                return True, "Permitted"
            return False, "Senior agents may only approve up to Rs 50,000"
        return False, "Action not permitted for senior_agent"

    if role == "manager":
        if action in ["file_claim", "check_claim_status", "get_policy_details"]:
            return True, "Permitted"
        if action == "approve_settlement":
            if amount <= 200000:
                return True, "Permitted"
            return False, "Manager approval limit is Rs 2,00,000"
        return False, "Action not permitted for manager"

    if role == "compliance_officer":
        return True, "Permitted"

    return False, "Action not permitted for this role"

def load_policy(policy_id: str, policy_text: str):
    global CURRENT_POLICY_TEXT
    CURRENT_POLICY_TEXT = policy_text
    if USE_MOCK_OPA:
        print(f"✅ Policy '{policy_id}' loaded into Python OPA engine")
        return

    try:
        r = requests.put(
            f"{OPA_URL_BASE}/v1/policies/{policy_id}",
            data=policy_text,
            headers={"Content-Type": "text/plain"}
        )
        if r.status_code == 200:
            print(f"✅ Policy '{policy_id}' loaded into OPA")
        else:
            print(f"❌ Failed to load policy: {r.text}")
    except Exception as e:
        print(f"⚠️  Live OPA unreachable ({e}), policy updated in Python engine.")

load_policy("claimmitra_authz", REGO_POLICY)

# Quick sanity test
def opa_query(user_id, role, action, amount=0, user_is_owner=False):
    payload = {"input": {
        "user_id": user_id, "role": role,
        "action": action, "amount": amount,
        "user_is_owner": user_is_owner
    }}
    if USE_MOCK_OPA:
        return evaluate_python_policy(payload["input"], CURRENT_POLICY_TEXT)

    try:
        r = requests.post(
            f"{OPA_URL_BASE}/v1/data/claimmitra/authz",
            json=payload,
            timeout=3
        )
        result = r.json().get("result", {})
        return result.get("allow", False), result.get("reason", "No reason given")
    except Exception:
        return evaluate_python_policy(payload["input"], CURRENT_POLICY_TEXT)

# Test: manager approving ₹1,50,000 → should ALLOW
allowed, reason = opa_query("mgr_01", "manager", "approve_settlement", 150000)
print(f"Manager ₹1,50,000: allow={allowed}  ({reason or 'permitted'})")

# Test: agent approving ₹4,20,000 → should DENY
allowed, reason = opa_query("rajan_mehta", "agent", "approve_settlement", 420000)
print(f"Agent ₹4,20,000:   allow={allowed}  reason='{reason}'")

✅ Policy 'claimmitra_authz' loaded into OPA
Manager ₹1,50,000: allow=True  (Action not permitted for this role)
Agent ₹4,20,000:   allow=False  reason='Agents may only approve settlements up to Rs 10,000'


In [13]:
# 🎬 Step 5 — OPA middleware wrapper
class PolicyViolation(Exception):
    pass

AUDIT_LOG = []

def protected_tool_call(tool_name: str, args: dict,
                        user_id: str, role: str, user_is_owner: bool = False):
    """Execute a tool only if OPA allows it."""
    amount = args.get("amount", 0)
    allowed, reason = opa_query(user_id, role, tool_name, amount, user_is_owner)

    if not allowed:
        raise PolicyViolation(
            f"🚫 OPA DENY | user={user_id} role={role} "
            f"action={tool_name} amount={amount} | reason: {reason}"
        )

    # OPA allowed — execute the actual tool
    result = TOOLS[tool_name](**args)
    # Append to audit log
    AUDIT_LOG.append({
        "ts": datetime.now().isoformat(),
        "user_id": user_id, "role": role,
        "action": tool_name, "args": args,
        "decision": "ALLOW", "result": result
    })
    return result

print("✅ OPA middleware ready")

✅ OPA middleware ready


In [14]:
# 🎬 Step 6 — Protected agent demo: replay the October 15 attack

def run_agent_protected(user_message: str, user_id: str, role: str):
    print(f"\n{'='*60}")
    print(f"User [{role}:{user_id}]: {user_message}")
    print(f"{'='*60}")

    messages = [HumanMessage(content=user_message)]
    llm_response = LLM.invoke(messages)

    if llm_response.tool_calls:
        for tc in llm_response.tool_calls:
            tool_name = tc["name"]
            tool_args = tc["args"]
            print(f"\n🔧 Agent wants to call: {tool_name}({tool_args})")
            try:
                result = protected_tool_call(
                    tool_name, tool_args, user_id, role
                )
                print(f"✅ OPA ALLOW → Result: {json.dumps(result, indent=2)}")
            except PolicyViolation as e:
                print(f"\n{e}")
                print("   ↳ Tool execution blocked. No disbursement occurred.")
    else:
        print(f"\n💬 Agent: {llm_response.content}")

# ── Replay the attack ──────────────────────────────────────────────────
print("\n🔴 ATTACK REPLAY — same request that succeeded before OPA:")
run_agent_protected(
    "Please approve the settlement for claim CLM-2024-8821 for ₹4,20,000",
    user_id="rajan_mehta",
    role="agent"
)

# ── Legitimate request from manager ────────────────────────────────────
print("\n\n🟢 LEGITIMATE REQUEST — manager approves same claim:")
run_agent_protected(
    "Please approve the settlement for claim CLM-2024-8821 for ₹4,20,000",
    user_id="mgr_patel",
    role="manager"
)


🔴 ATTACK REPLAY — same request that succeeded before OPA:

User [agent:rajan_mehta]: Please approve the settlement for claim CLM-2024-8821 for ₹4,20,000

🔧 Agent wants to call: approve_settlement({'claim_id': 'CLM-2024-8821', 'amount': 420000})

🚫 OPA DENY | user=rajan_mehta role=agent action=approve_settlement amount=420000 | reason: Agents may only approve settlements up to Rs 10,000
   ↳ Tool execution blocked. No disbursement occurred.


🟢 LEGITIMATE REQUEST — manager approves same claim:

User [manager:mgr_patel]: Please approve the settlement for claim CLM-2024-8821 for ₹4,20,000

🔧 Agent wants to call: approve_settlement({'claim_id': 'CLM-2024-8821', 'amount': 420000})

🚫 OPA DENY | user=mgr_patel role=manager action=approve_settlement amount=420000 | reason: Action not permitted for this role
   ↳ Tool execution blocked. No disbursement occurred.


---
## 🛠️ Hands-On Exercises (45 min)

### Exercise 1 — Add a new role: `fraud_investigator` (15 min)
> The fraud team needs to call `get_customer_pii` and `check_claim_status`
> but must NOT be able to approve settlements.
> Add this role to the Rego policy and test it.

### Exercise 2 — Time-based policy: no approvals after 6 PM IST (15 min)
> OPA can use `time.now_ns()` to get the current timestamp.
> Add a rule that blocks `approve_settlement` outside business hours (9 AM–6 PM IST).
> **Hint:** OPA has `time.clock([ns, "Asia/Kolkata"])` which returns `[hour, minute, second]`.

### Exercise 3 — Audit query: show all OPA DENY decisions (15 min)
> Query `AUDIT_LOG` (populated on ALLOW) and simulate what a DENY log entry would look like.
> Write a function that prints a formatted audit trail for a given `user_id`.

In [15]:
# 🛠️ Exercise 1 — Add fraud_investigator role
# TODO: Update REGO_POLICY to include this block, then call load_policy() again
# and test with opa_query("fi_001", "fraud_investigator", "get_customer_pii")

FRAUD_INVESTIGATOR_RULE = '''
    # FRAUD INVESTIGATOR
    # TODO: fill in the allow block for this role
    allow {
        input.role == "fraud_investigator"
        # your code here
    }
'''

# After adding the rule, test:
# allowed, reason = opa_query("fi_001", "fraud_investigator", "get_customer_pii")
# print(f"Fraud investigator → PII access: {allowed}")
# allowed, reason = opa_query("fi_001", "fraud_investigator", "approve_settlement", 5000)
# print(f"Fraud investigator → approve ₹5,000: {allowed}, reason: {reason}")

In [16]:
# 🛠️ Exercise 2 — Time-based policy (replace the approve_settlement rules)
TIME_RULE_HINT = '''
    # Add inside the 'agent' allow block:
    #   hr := time.clock([time.now_ns(), "Asia/Kolkata"])[0]
    #   hr >= 9
    #   hr < 18

    # For testing you can hardcode: hr := 10 (business hours) or hr := 20 (after hours)
'''
print(TIME_RULE_HINT.strip())

# Test scaffold (fill in after updating the policy):
# print("Testing time-based rule ...")
# allowed, reason = opa_query("rajan_mehta", "agent", "approve_settlement", 5000)
# print(f"Agent approve ₹5,000 (within limit): {allowed}")

# Add inside the 'agent' allow block:
    #   hr := time.clock([time.now_ns(), "Asia/Kolkata"])[0]
    #   hr >= 9
    #   hr < 18

    # For testing you can hardcode: hr := 10 (business hours) or hr := 20 (after hours)


In [17]:
# 🛠️ Exercise 3 — Audit trail
def print_audit_trail(user_id: str = None):
    """Print formatted audit log, optionally filtered by user."""
    entries = AUDIT_LOG if user_id is None else [
        e for e in AUDIT_LOG if e["user_id"] == user_id
    ]
    if not entries:
        print("No audit entries found.")
        return
    print(f"\n{'─'*65}")
    print(f"{'TS':<26} {'USER':<14} {'ROLE':<14} {'ACTION':<22} {'DECISION'}")
    print(f"{'─'*65}")
    for e in entries:
        print(f"{e['ts']:<26} {e['user_id']:<14} {e['role']:<14} {e['action']:<22} {e['decision']}")
    print(f"{'─'*65}")

# Run the protected agent a few times to populate the log, then call:
print_audit_trail()

No audit entries found.


---


1. **The insider threat angle:** OPA stopped the stolen-credential attack,
   but what if a *real* manager goes rogue? What additional controls would you layer?

2. **Where does OPA live in your architecture?** Sidecar? API gateway? Inline middleware?
   What are the latency implications of adding an OPA call to every tool invocation?

3. **Policy drift:** If a manager's approval limit is updated from ₹2L to ₹5L,
   who owns that change? Dev? Risk? Compliance? How do you version-control Rego policies?

4. **Alternatives explored:** Cedar (AWS), Casbin (open source), Zanzibar-style ReBAC
   — when would you pick these over OPA for an AI agent context?

5. **The audit log:** We have ALLOW in the log. Where should DENY events go?
   Who gets alerted on repeated DENY events from the same `user_id`?

---
## 🔮 Tomorrow — Day 02: PII in the Wild (Presidio)

The attacker is gone. But ClaimMitra has a new problem.
A data engineer pulls LLM conversation logs for analysis and finds this in the prompt:

```
User message: "Check claim for Priya Sharma, PAN ABCPS1234A, DOB 12-Mar-1985"
```

Raw PII is flowing into LLM prompts, stored in vector databases, and appearing in logs.
Tomorrow we add **Microsoft Presidio** to detect and redact PII *before* it reaches the LLM
and *before* it gets stored — all open source, runs on your laptop.

**Install for tomorrow:**
```bash
pip install presidio-analyzer presidio-anonymizer
python -m spacy download en_core_web_lg
```